# RandomForestClassifier on `hab_ndbc_merged.csv`

This notebook builds a bloom classifier on `hab_ndbc_merged.csv` and focuses on how the `class_weight` parameter changes model behavior.

Goals:
- predict `potential_bloom`
- keep the forest hyperparameters fixed so the comparison stays centered on `class_weight`
- compare recall, precision, F1, balanced accuracy, ROC AUC, and average precision
- avoid leakage by excluding `pda` and `isHarmful`, which directly encode the target

Notes:
- `potential_bloom` is strongly imbalanced in this dataset
- the final holdout split is chronological so newer observations stay in the test set
- if `scikit-learn` is missing in your environment, install it before running the modeling cells


In [ ]:
# If needed, uncomment and run:
# %pip install scikit-learn matplotlib pandas numpy

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")


In [ ]:
DATA_PATH = Path("../hab_ndbc_merged.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("hab_ndbc_merged.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["week_start", "sample_date"])
df = df.sort_values("week_start").reset_index(drop=True)

print(f"Loaded {DATA_PATH}")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
target_col = "potential_bloom"

target_summary = pd.DataFrame(
    {
        "count": df[target_col].value_counts().sort_index(),
        "rate": df[target_col].value_counts(normalize=True).sort_index(),
    }
)

print("Target distribution:")
display(target_summary)

print("Top missing columns:")
display(df.isna().sum().sort_values(ascending=False).head(12).to_frame("missing_rows"))

## Feature setup

The target is `potential_bloom`. We intentionally drop:
- `pda` because `potential_bloom` was derived from it
- `isHarmful` because it duplicates the target in this file
- `week_start` and `sample_date` from the feature matrix because `month` and `year` already provide simple temporal context, and we use `week_start` only for splitting
- `station_id` because `station` already represents site identity without imposing an ordinal relationship


In [ ]:
numeric_features = [
    "latitude",
    "longitude",
    "month",
    "year",
    "temp",
    "silicate",
    "nitrate",
    "avg_chloro",
    "sst_roll_14d",
    "anom_roll_14d",
    "sst_roc_3d",
    "warm_degree_days_14d",
    "above_avg",
    "temp_lag1",
    "temp_lag2",
    "silicate_lag1",
    "silicate_lag2",
    "nitrate_lag1",
    "nitrate_lag2",
    "avg_chloro_lag1",
    "avg_chloro_lag2",
    "silicate_nitrate_ratio",
    "wind_speed_mps",
    "wave_height_m",
    "dominant_period_s",
    "mean_wave_dir_deg",
    "atm_pressure_hpa",
    "air_temp_c",
    "sea_surface_temp_c",
]

categorical_features = ["station"]

feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df[target_col].astype(int).copy()

print(f"Feature columns: {len(feature_cols)}")
display(X.head())

In [ ]:
split_idx = int(len(df) * 0.80)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_test)],
        "positive_count": [int(y_train.sum()), int(y_test.sum())],
        "positive_rate": [y_train.mean(), y_test.mean()],
        "start_week": [df.loc[: split_idx - 1, "week_start"].min(), df.loc[split_idx:, "week_start"].min()],
        "end_week": [df.loc[: split_idx - 1, "week_start"].max(), df.loc[split_idx:, "week_start"].max()],
    },
    index=["train", "test"],
)

imbalance_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"Train negative/positive ratio: {imbalance_ratio:0.2f}:1")
display(split_summary)

In [ ]:
def build_pipeline(class_weight):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
            (
                "numeric",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                numeric_features,
            ),
        ]
    )

    model = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight=class_weight,
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


class_weight_options = {
    "none": None,
    "balanced": "balanced",
    "balanced_subsample": "balanced_subsample",
    "positive_x5": {0: 1.0, 1: 5.0},
    "positive_x10": {0: 1.0, 1: 10.0},
    "positive_x20": {0: 1.0, 1: 20.0},
}

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
class_weight_options

## Cross-validated comparison on the training window

We use stratified CV inside the training period to make the class-weight comparison stable, then keep a newer chronological holdout set for the final evaluation. The winning `class_weight` is chosen from CV results, not from the holdout set.

In [ ]:
cv_rows = []

for label, weight in class_weight_options.items():
    pipeline = build_pipeline(weight)
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    cv_rows.append(
        {
            "class_weight_label": label,
            "class_weight": str(weight),
            **{
                f"cv_{metric}": float(np.mean(scores[f"test_{metric}"]))
                for metric in scoring
            },
        }
    )

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["cv_f1", "cv_recall", "cv_average_precision"],
    ascending=False,
).reset_index(drop=True)

cv_results

In [ ]:
test_rows = []
fitted_models = {}

for label, weight in class_weight_options.items():
    pipeline = build_pipeline(weight)
    pipeline.fit(X_train, y_train)
    fitted_models[label] = pipeline

    y_pred = pipeline.predict(X_test)
    y_score = pipeline.predict_proba(X_test)[:, 1]

    test_rows.append(
        {
            "class_weight_label": label,
            "test_precision": precision_score(y_test, y_pred, zero_division=0),
            "test_recall": recall_score(y_test, y_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_pred, zero_division=0),
            "test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "test_roc_auc": roc_auc_score(y_test, y_score),
            "test_average_precision": average_precision_score(y_test, y_score),
            "predicted_positives": int(y_pred.sum()),
        }
    )

test_results = pd.DataFrame(test_rows)

comparison = cv_results.merge(test_results, on="class_weight_label").sort_values(
    ["test_f1", "test_recall", "test_average_precision"],
    ascending=False,
).reset_index(drop=True)

comparison

In [ ]:
plot_cols = ["test_precision", "test_recall", "test_f1", "test_balanced_accuracy"]
plot_df = comparison.set_index("class_weight_label")[plot_cols]

ax = plot_df.plot(kind="bar", figsize=(12, 5))
ax.set_title("Holdout metrics by class_weight")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Fit and inspect the best class-weight setting

The notebook selects the best candidate from cross-validation using F1 first, then recall and average precision as tie-breakers. If your priority is higher bloom recall, you can change the sorting rule below.

In [ ]:
best_label = cv_results.sort_values(
    ["cv_f1", "cv_recall", "cv_average_precision"],
    ascending=False,
).iloc[0]["class_weight_label"]
best_model = fitted_models[best_label]

print(f"Selected class_weight setting: {best_label}")
print(comparison.loc[comparison["class_weight_label"] == best_label])

y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, cmap="Blues")
plt.title(f"Confusion matrix: {best_label}")
plt.tight_layout()
plt.show()

In [ ]:
feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
importances = best_model.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top_features = importance_df.head(15).sort_values("importance")
ax = top_features.plot(kind="barh", x="feature", y="importance", figsize=(10, 6), legend=False)
ax.set_title(f"Top feature importances for {best_label}")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

importance_df.head(15)

## Next steps

Useful follow-ups if you want to push the analysis further:
- tune `min_samples_leaf`, `max_depth`, and `n_estimators` after choosing a promising `class_weight`
- compare against `BalancedRandomForestClassifier` from `imbalanced-learn`
- try probability threshold tuning if recall matters more than default 0.5 classification
- swap the split strategy to station-based or rolling-time validation if you want a stricter generalization test
